Name: Moksh Barapatre

Subject: MLDS LAB

Section: B3

Batch: B3

Roll no.: 49

# MLDS Project: AI-Based Predictive Maintenance

This project predicts the `Machine failure` column of the AI4I 2020 dataset. The inputs are product type and five sensor readings. The failure-mode columns `TWF`, `HDF`, `PWF`, `OSF`, and `RNF` are not inputs for the overall model because they directly describe the failure and would leak the answer.

A second model uses the same inputs to estimate the five failure modes individually. Here the mode columns are the labels, so the model still has to learn the pattern from the sensor readings.

The notebook follows the same steps as the practicals: load the data, inspect it, plot it, preprocess it, train three models, evaluate them, tune the Random Forest, train a multi-label failure-mode model, and save the final models for the Streamlit app.

## 1. Import Libraries

We need pandas for data handling, matplotlib and seaborn for plots, and scikit-learn for preprocessing, models, and metrics.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

sns.set_style("whitegrid")

In [ ]:
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    classification_report,
)

## 2. Load Dataset

The CSV is in the `data` folder. The small path check lets the notebook run from the project root or from the `notebooks` folder.

In [ ]:
data_path = Path("data/ai4i2020.csv")
if not data_path.exists():
    data_path = Path("../data/ai4i2020.csv")

df = pd.read_csv(data_path)

target = "Machine failure"
failure_modes = ["TWF", "HDF", "PWF", "OSF", "RNF"]

# The overall model must not use these columns as inputs.
leakage_columns = failure_modes

df.head()

## 3. Data Inspection

We check the shape, data types, summary statistics, missing values, duplicates, and class balance before modeling.

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
df.describe()

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
print("Class counts:")
print(df[target].value_counts())

print("\nFailure rate:", round(df[target].mean() * 100, 2), "%")

## 4. Exploratory Data Analysis

The target is imbalanced: most records are normal and only a small number are failures. The plots below show the class counts, feature distributions, correlations, and group averages.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=target)
plt.title("Machine Failure Counts")
plt.xlabel("Machine failure (0 = no, 1 = yes)")
plt.ylabel("Records")
plt.show()

In [ ]:
numeric_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, column in zip(axes.flat, numeric_features):
    sns.histplot(data=df, x=column, hue=target, bins=30, ax=ax)
    ax.set_title(column)

fig.delaxes(axes.flat[5])
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, column in zip(axes, ["Air temperature [K]", "Torque [Nm]", "Tool wear [min]"]):
    sns.boxplot(data=df, x=target, y=column, ax=ax)
    ax.set_title(column)

plt.tight_layout()
plt.show()

In [ ]:
df.groupby(target)[numeric_features].mean().T

The failure class is much smaller than the normal class. Torque and tool wear are higher in failure cases, while rotational speed is lower on average. The ranges still overlap, so the model needs to combine several features.

## 5. Data Preprocessing

We use `Type` and the five numeric sensor columns as inputs. `Type` is one-hot encoded and the numeric columns are standardized. The preprocessing is fitted on the training data only, then applied to the test data.

The failure-mode columns are not inputs here. They are used later as labels for a separate failure-mode model.

In [ ]:
categorical_features = ["Type"]
features = categorical_features + numeric_features

X = df[features]
y = df[target]

print("Features:", features)
print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train failure rate:", round(y_train.mean() * 100, 2), "%")
print("Test failure rate:", round(y_test.mean() * 100, 2), "%")

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
], verbose_feature_names_out=False)

# Fit only on the training data, then transform the test data.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)
print("Feature names:", preprocessor.get_feature_names_out())

## 6. Train Models

We train three classifiers with class weights because failures are rare. Each model uses the processed training data and is evaluated on the processed test data.

In [ ]:
log_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42,
)
log_model.fit(X_train_processed, y_train)
log_pred = log_model.predict(X_test_processed)

In [ ]:
tree_model = DecisionTreeClassifier(
    class_weight="balanced",
    min_samples_leaf=5,
    random_state=42,
)
tree_model.fit(X_train_processed, y_train)
tree_pred = tree_model.predict(X_test_processed)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=160,
    class_weight="balanced_subsample",
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42,
)
rf_model.fit(X_train_processed, y_train)
rf_pred = rf_model.predict(X_test_processed)

In [ ]:
models = {
    "Logistic Regression": (log_model, log_pred),
    "Decision Tree": (tree_model, tree_pred),
    "Random Forest": (rf_model, rf_pred),
}

In [ ]:
rows = []
for name, (model, pred) in models.items():
    prob = model.predict_proba(X_test_processed)[:, 1]
    rows.append([
        name,
        accuracy_score(y_test, pred),
        precision_score(y_test, pred, zero_division=0),
        recall_score(y_test, pred, zero_division=0),
        f1_score(y_test, pred, zero_division=0),
        roc_auc_score(y_test, prob),
    ])

comparison = pd.DataFrame(
    rows,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
).set_index("Model")

comparison.round(4)

Logistic Regression catches more failures but has many false alarms. The Random Forest has better precision and the highest ROC-AUC, so it gives the best overall ranking of failure risk.

## 7. Evaluate Models

A confusion matrix shows how many failures were missed. A false negative means a failure was predicted as normal, which is costly in maintenance. ROC-AUC shows how well the model ranks failures across all thresholds.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, (model, pred)) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=ax,
        xticklabels=["No", "Fail"],
        yticklabels=["No", "Fail"],
    )
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
for name, (model, pred) in models.items():
    prob = model.predict_proba(X_test_processed)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, label=name)

plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.show()

In [ ]:
for name, (model, pred) in models.items():
    print(f"\n{name}")
    print(classification_report(
        y_test,
        pred,
        target_names=["No failure", "Failure"],
        zero_division=0,
    ))

## 8. Cross-Validation and Tuning

We use 5-fold stratified cross-validation on the training data. The Random Forest pipeline is tuned with a small grid search. Recall is the selection metric because missing a failure is more costly than a false alarm.

In [ ]:
rf_preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
], verbose_feature_names_out=False)

In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", rf_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=160,
        class_weight="balanced_subsample",
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42,
    )),
])

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_cv_recall = cross_val_score(rf_pipeline, X_train, y_train, cv=cv, scoring="recall")
rf_cv_f1 = cross_val_score(rf_pipeline, X_train, y_train, cv=cv, scoring="f1")
rf_cv_auc = cross_val_score(rf_pipeline, X_train, y_train, cv=cv, scoring="roc_auc")

print("Baseline RF CV recall:", round(rf_cv_recall.mean(), 4))
print("Baseline RF CV F1:", round(rf_cv_f1.mean(), 4))
print("Baseline RF CV ROC-AUC:", round(rf_cv_auc.mean(), 4))

In [ ]:
param_grid = {
    "classifier__n_estimators": [120, 200],
    "classifier__max_depth": [None, 12],
    "classifier__min_samples_leaf": [1, 3],
}

In [ ]:
search = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=cv,
    scoring="recall",
)

search.fit(X_train, y_train)
best_model = search.best_estimator_

print("Best parameters:", search.best_params_)
print("Best CV recall:", round(search.best_score_, 4))

In [ ]:
tuned_cv_recall = cross_val_score(best_model, X_train, y_train, cv=cv, scoring="recall")
tuned_cv_f1 = cross_val_score(best_model, X_train, y_train, cv=cv, scoring="f1")
tuned_cv_auc = cross_val_score(best_model, X_train, y_train, cv=cv, scoring="roc_auc")

cv_table = pd.DataFrame([
    ["Baseline RF", rf_cv_recall.mean(), rf_cv_f1.mean(), rf_cv_auc.mean()],
    ["Tuned RF", tuned_cv_recall.mean(), tuned_cv_f1.mean(), tuned_cv_auc.mean()],
], columns=["Model", "CV Recall", "CV F1", "CV ROC-AUC"]).set_index("Model")

cv_table.round(4)

The tuned forest has higher cross-validated recall than the baseline forest. This means it catches more failures on unseen folds, which is the main goal for maintenance screening.

In [ ]:
best_pred = best_model.predict(X_test)
best_prob = best_model.predict_proba(X_test)[:, 1]

best_metrics = {
    "Accuracy": accuracy_score(y_test, best_pred),
    "Precision": precision_score(y_test, best_pred, zero_division=0),
    "Recall": recall_score(y_test, best_pred, zero_division=0),
    "F1": f1_score(y_test, best_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, best_prob),
}

best_metrics

## 9. Feature Importance

The tuned Random Forest gives an importance score for each transformed feature. This shows which inputs the model used most, but it does not prove cause and effect.

In [ ]:
best_preprocessor = best_model.named_steps["preprocessor"]
best_classifier = best_model.named_steps["classifier"]

importance = pd.DataFrame({
    "Feature": best_preprocessor.get_feature_names_out(),
    "Importance": best_classifier.feature_importances_,
}).sort_values("Importance", ascending=False)

importance

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=importance, x="Importance", y="Feature")
plt.title("Random Forest Feature Importance")
plt.show()

Rotational speed, torque, and tool wear are the most important features. This matches the patterns seen in the exploratory plots.

## 10. Failure Mode Analysis

The overall model answers one question: will the machine fail? We can also train a second model to estimate which failure mode is most likely. The five targets are `TWF`, `HDF`, `PWF`, `OSF`, and `RNF`.

This is a multi-label problem because one record can have more than one failure mode. The mode columns are not inputs. They are used only as labels, so the model learns from the sensor readings.

In [ ]:
mode_counts = df[failure_modes].sum().sort_values(ascending=False)
mode_counts

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(x=mode_counts.index, y=mode_counts.values)
plt.title("Failure Mode Counts")
plt.ylabel("Records")
plt.show()

Every mode is rare. `RNF` has the fewest positive records, so its test scores will be less stable. This is why we check each mode separately instead of averaging all five into one score.

In [ ]:
mode_train = df.loc[X_train.index, failure_modes]
mode_test = df.loc[X_test.index, failure_modes]

print("Mode train shape:", mode_train.shape)
print("Mode test shape:", mode_test.shape)

In [ ]:
mode_preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
], verbose_feature_names_out=False)

In [ ]:
mode_classifier = MultiOutputClassifier(RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced_subsample",
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42,
))

mode_pipeline = Pipeline([
    ("preprocessor", mode_preprocessor),
    ("classifier", mode_classifier),
])

In [ ]:
mode_pipeline.fit(X_train, mode_train)
mode_probabilities = mode_pipeline.predict_proba(X_test)

In [ ]:
mode_rows = []
for i, mode in enumerate(failure_modes):
    probability = mode_probabilities[i][:, 1]
    prediction = (probability >= 0.5).astype(int)
    if mode_test[mode].nunique() == 2:
        auc = roc_auc_score(mode_test[mode], probability)
    else:
        auc = float("nan")
    mode_rows.append([
        mode,
        int(mode_test[mode].sum()),
        precision_score(mode_test[mode], prediction, zero_division=0),
        recall_score(mode_test[mode], prediction, zero_division=0),
        f1_score(mode_test[mode], prediction, zero_division=0),
        auc,
        average_precision_score(mode_test[mode], probability),
    ])

mode_metrics = pd.DataFrame(
    mode_rows,
    columns=["Mode", "Test positives", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"],
).set_index("Mode")

mode_metrics.round(3)

The model learns separate patterns for each mode. HDF, PWF, and OSF have more positive examples, so their scores are usually stronger. TWF and RNF are much harder, and a high probability for them should be treated as a rough signal, not a diagnosis.

The app uses this saved pipeline to show five independent probabilities after a prediction. They do not need to add to 100%, because more than one mode can occur together.

## 11. Save Final Models

The tuned overall pipeline is saved for the Streamlit app. It includes both the preprocessing and the Random Forest, so the app can pass raw input values directly. The failure-mode pipeline is saved in the same way. We also save the feature order, test metrics, model comparison, feature importance, and failure-mode metrics.

In [ ]:
project_root = data_path.parent.parent
model_dir = project_root / "models"
model_dir.mkdir(exist_ok=True)

with open(model_dir / "best_model.pkl", "wb") as f: pickle.dump(best_model, f)
with open(model_dir / "failure_mode_model.pkl", "wb") as f: pickle.dump(mode_pipeline, f)
print("Saved models.")

In [ ]:
metadata = {
    "target": target,
    "features": features,
    "feature_order": features,
    "excluded_leakage_columns": leakage_columns,
    "best_params": search.best_params_,
    "test_metrics": {key: float(value) for key, value in best_metrics.items()},
    "test_size": 0.2,
    "random_state": 42,
    "cv_folds": 5,
    "selection_metric": "recall",
    "dataset": "UCI AI4I 2020 Predictive Maintenance Dataset",
}

mode_metadata = {
    "targets": failure_modes,
    "target_order": failure_modes,
    "features": features,
    "feature_order": features,
    "test_size": 0.2,
    "random_state": 42,
    "dataset": "UCI AI4I 2020 Predictive Maintenance Dataset",
}

(model_dir / "model_metadata.json").write_text(json.dumps(metadata, indent=2))
(model_dir / "failure_mode_metadata.json").write_text(json.dumps(mode_metadata, indent=2))
print("Saved metadata.")

In [ ]:
comparison.to_csv(model_dir / "model_comparison.csv")
importance.to_csv(model_dir / "feature_importance.csv", index=False)
mode_metrics.to_csv(model_dir / "failure_mode_metrics.csv")

print("Saved result tables.")

## 12. Conclusion

The tuned Random Forest is the final overall failure model. A separate multi-label forest uses the same sensor inputs to estimate TWF, HDF, PWF, OSF, and RNF individually. The app shows both the overall risk and the failure-mode estimates.

The main limitation is the class imbalance. Failures are rare, and some modes such as TWF and RNF have very few positive examples, so their estimated probabilities are less reliable. The dataset is also synthetic. A real system would need site-specific data, threshold tuning, calibration, and validation before use.